In [49]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [50]:
train= pd.read_csv("../data/train.csv")
test= pd.read_csv("../data/test.csv")
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [51]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

train["Deck"] = train["Cabin"].str[0]

test["Deck"] = test["Cabin"].str[0]



train["Deck"] = train["Deck"].fillna("Unknown")

test["Deck"] = test["Deck"].fillna("Unknown")




In [52]:
train.drop(
    columns=["PassengerId", "Name", "Ticket", "Cabin"],
    inplace=True
)

test.drop(
    columns=["PassengerId", "Name", "Ticket", "Cabin"],
    inplace=True
)

In [53]:
y= train["Survived"]
X= train.drop(columns="Survived")
categorical_features = X.select_dtypes(include=["object"]).columns

numerical_features = X.select_dtypes(exclude=["object"]).columns

print(categorical_features)
print(numerical_features)

Index(['Sex', 'Embarked', 'Deck'], dtype='object')
Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone'], dtype='object')


In [54]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_tranformer= Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor= ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat",categorical_tranformer, categorical_features)
])
print(train_test_split)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=0)

<function train_test_split at 0x000001B63A4FE710>


In [55]:
from xgboost import XGBClassifier

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        learning_rate=0.03,
        max_depth=10,
        n_estimators=1000
    ))
])

model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [56]:
pred = model.predict(X_valid)
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_valid, pred)

print(accuracy)

0.8659217877094972
